# LettuceDetect baseline — Hallucination Detection in Tool Calling

Evaluates the off-the-shelf `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint on three test splits we built from ToolACE, each corresponding to one hallucination type:

- **Type 1** — *Hallucination*: answer contradicts tool output.
- **Type 2** — *Overgeneration*: answer adds facts not in tool output.
- **Type 3** — *Missing tool*: answer proposes an action requiring an unavailable tool.

For each split we report span-level (micro + macro F1) and example-level (P/R/F1) metrics.

**Requires a GPU runtime (T4 / P100 / L4).** Inference on ~150 samples takes ~30 s on T4.

Upload the test files (`test.jsonl`, `type2_test.jsonl`, `type3_test.jsonl`) as a Kaggle dataset and point `DATA_DIR` to its mount path below.


In [ ]:
import subprocess, sys, os, importlib.util

NEEDED = ("lettucedetect",)
missing = [p for p in NEEDED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    # Colab pre-loads older numpy/transformers; restart the kernel so new versions are picked up.
    if "google.colab" in sys.modules:
        print("Restarting kernel to pick up new packages — re-run the notebook from the top after restart.")
        os.kill(os.getpid(), 9)
else:
    print("lettucedetect already installed.")


## Setup

This notebook runs in **Google Colab** or **Kaggle**. Steps below:

1. **Enable GPU**: `Runtime → Change runtime type → T4 GPU` (Colab) or `Settings → Accelerator → GPU` (Kaggle).
2. **Upload the three test files** (`test.jsonl`, `type2_test.jsonl`, `type3_test.jsonl`):
   - **Colab**: click the folder icon in the left sidebar → upload button → drop the three files into `/content/`. (Or run the upload cell below.)
   - **Kaggle**: upload them as a Kaggle dataset, then point `DATA_DIR` at `/kaggle/input/<your-dataset>/`.
3. **Run all cells**.

The expected filenames are `test.jsonl` (Type 1), `type2_test.jsonl`, `type3_test.jsonl`.

In [ ]:
from pathlib import Path

CANDIDATE_DIRS = [
    Path("/content"),
    Path("/kaggle/input/halluc-toolace"),
    Path("data"),
    Path("."),
]
REQUIRED = ["combined_test.jsonl"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if all((d / f).exists() for f in REQUIRED)), None)

if DATA_DIR is None:
    print("combined_test.jsonl not found. Upload it via the cell below.")
else:
    print(f"Using DATA_DIR = {DATA_DIR}")

MODEL_PATH = "KRLabsOrg/lettucedect-large-modernbert-en-v1"


### Optional: upload via Colab UI

If you didn't drop the files into `/content/` manually, run this cell — it will pop up a file picker. (Skip if `DATA_DIR` is already set above.)

In [ ]:
if DATA_DIR is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_DIR = Path("/content")
        print(f"Uploaded: {list(uploaded.keys())}")
        print(f"DATA_DIR = {DATA_DIR}")
    except ImportError:
        print("Not in Colab. Set DATA_DIR manually or place combined_test.jsonl in one of the candidate dirs.")

target = DATA_DIR / "combined_test.jsonl"
print(f"combined_test.jsonl: {target}  exists={target.exists()}")


## Data loading and metrics

In [ ]:
import json
from dataclasses import dataclass


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def _char_set(spans):
    chars = set()
    for s in spans:
        chars.update(range(int(s["start"]), int(s["end"])))
    return chars


@dataclass
class Metrics:
    precision: float
    recall: float
    f1: float


def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample["hallucination_labels"])
        pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap
        micro_pred += len(pred)
        micro_gold += len(gold)
        if not pred and not gold:
            macro_f1.append(1.0); continue
        p = overlap / len(pred) if pred else 0.0
        r = overlap / len(gold) if gold else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        macro_f1.append(f1)
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    f_ma = sum(macro_f1) / len(macro_f1) if macro_f1 else 0.0
    return Metrics(p_mi, r_mi, f_mi), f_ma


def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample["hallucination_labels"] else 0
        pred = 1 if preds else 0
        if gold == 1 and pred == 1: tp += 1
        elif gold == 1 and pred == 0: fn += 1
        elif gold == 0 and pred == 1: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return Metrics(p, r, f1)


## Load LettuceDetect

This downloads the model (~400 MB, ModernBERT large) on first call.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model device: {next(detector.model.parameters()).device}")


In [ ]:
from lettucedetect.models.inference import HallucinationDetector

detector = HallucinationDetector(method="transformer", model_path=MODEL_PATH)
print(f"Loaded {MODEL_PATH}")


## Inference and metrics

In [ ]:
from tqdm.auto import tqdm

def predict(sample):
    raw = detector.predict(
        context=[sample["context"]],
        question=sample["query"],
        answer=sample["output"],
        output_format="spans",
    )
    return [
        {"start": int(s["start"]), "end": int(s["end"]),
         "text": s.get("text", sample["output"][int(s["start"]):int(s["end"])]),
         "confidence": float(s.get("confidence", 0.0))}
        for s in raw
    ]

# Inference once on combined_test (599 samples = 150 T1 + 150 T2 + 149 T3 + 150 clean).
samples_all = read_jsonl(DATA_DIR / "combined_test.jsonl")
preds_all = [predict(s) for s in tqdm(samples_all, desc="Inference (combined_test)")]

def filter_subset(type_label):
    """Return (samples, preds) restricted to samples of `type_label` plus all clean samples."""
    sel_s, sel_p = [], []
    for s, p in zip(samples_all, preds_all):
        if not s["hallucination_labels"]:
            sel_s.append(s); sel_p.append(p)
            continue
        if s["hallucination_labels"][0].get("type") == type_label:
            sel_s.append(s); sel_p.append(p)
    return sel_s, sel_p

subsets = {
    "Combined (all types + clean)":   (samples_all, preds_all),
    "Type 1 + clean":                 filter_subset("Type1_Contradiction"),
    "Type 2 + clean":                 filter_subset("Type2_Overgeneration"),
    "Type 3 + clean":                 filter_subset("Type3_MissingTool"),
}

results = {}
for name, (s, p) in subsets.items():
    span_micro, span_macro_f1 = span_metrics(s, p)
    ex = example_metrics(s, p)
    results[name] = {
        "n": len(s),
        "span_micro": span_micro,
        "span_macro_f1": span_macro_f1,
        "example": ex,
        "preds": p,
        "samples": s,
    }
    print(f"\n{name} (N={len(s)}): "
          f"span micro P/R/F1 = {span_micro.precision:.3f} / {span_micro.recall:.3f} / {span_micro.f1:.3f} | "
          f"span macro F1 = {span_macro_f1:.3f} | "
          f"example P/R/F1 = {ex.precision:.3f} / {ex.recall:.3f} / {ex.f1:.3f}")


## Summary table

In [ ]:
import pandas as pd

rows = []
for name, r in results.items():
    rows.append({
        "Split": name,
        "N": r["n"],
        "Span micro P": round(r["span_micro"].precision, 3),
        "Span micro R": round(r["span_micro"].recall, 3),
        "Span micro F1": round(r["span_micro"].f1, 3),
        "Span macro F1": round(r["span_macro_f1"], 3),
        "Ex P": round(r["example"].precision, 3),
        "Ex R": round(r["example"].recall, 3),
        "Ex F1": round(r["example"].f1, 3),
    })
df = pd.DataFrame(rows)
df


## Qualitative inspection

Show a few samples per split with gold span vs predicted spans.

In [ ]:
def show(sample, preds, n=3, width=120):
    out = sample["output"]
    gold = sample["hallucination_labels"]
    print(f"--- {sample.get('id', '?')} ---")
    for g in gold:
        s, e = g["start"], g["end"]
        print(f"  GOLD: [{s}..{e}] {out[s:e]!r}")
    if preds:
        for p in preds:
            s, e = p["start"], p["end"]
            print(f"  PRED: [{s}..{e}] {out[s:e]!r}  (conf={p.get('confidence', 0):.2f})")
    else:
        print(f"  PRED: (none)")
    print()

for name, r in results.items():
    print(f"\n=== {name} ===")
    for i in range(min(3, r["n"])):
        show(r["samples"][i], r["preds"][i])
